<a id="unit-u.383ee093446d4490989be00fdf176f5f"></a>

<div>APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ · ҚОСЫМША ЗЕРТХАНАЛЫҚ ЖҰМЫС</div>

<a id="optional-lab-5b-mysql--flink-cdc--doris"></a>

# Қосымша 5B жұмысы: MySQL → Flink CDC → Doris

**5-модуль: деректерді пакеттік және үздіксіз жүктеу** · [Сабақ](course.md) · [Тест](quiz05_load_methods_and_retry_safety.ipynb) · [Оқу: үздіксіз жүктеу](optional05_continuous_ingestion.md)

Flink тапсырмасы MySQL CDC коннекторымен MySQL кестесін оқиды: алдымен бар жолдарды (**snapshot**), содан кейін MySQL binlog ішіндегі әр өзгерісті, әрі Doris коннекторымен Doris-ке жазады. Snapshot-ты тексересіз, дереккөзді өзгертіп, енгізулер, жаңартулар мен жоюлардың жетуін бақылайсыз, дереккөз өзгеріп жатқанда тапсырманы savepoint-пен тоқтатасыз және қуып жетуі үшін savepoint-тен қалпына келтіресіз. 7-модуль сол идеяларды қайта ойнатылған оқиғалармен дамытады.

Шамамен 40 минут жоспарлаңыз. Жұмыс таңдау бойынша: басқа жұмыстар оған тәуелді емес.

**Не қажет**

- Кемінде 10 GiB жады және 4 CPU бар Docker (Docker Desktop: Settings → Resources). Бұл жұмыста Apple silicon бар Docker Desktop-та өлшенген ең жоғары пайдалану: Doris 2.8 GiB, Flink JobManager 1.8 GiB, TaskManager 1.1 GiB, MySQL 0.5 GiB. Бірінші ұяшық ештеңе іске қоспас бұрын Docker ресурстарын тексереді.
- Алғашқы жүктеулер: `mysql:8.0.33` (шамамен 800 MB), `flink:1.20.1-scala_2.12-java11` және Maven Central-дан үш коннектор JAR-ы (шамамен 65 MB); Compose оларды шамамен бір минутта `doris-course-flink` (1.5 GB) курс бейнесіне жинайды.
- 1-зертханалық жұмыстың курс сынақ ортасы. Бірінші ұяшық Doris контейнерін 6 GiB жады шегімен қайта жасайды; деректері сақталады. Алдымен басқа ноутбуктерді аяқтаңыз. 1-зертханалық жұмыстың **Іске қосу және қосылу** ұяшығы бұл шекті қайта алып тастайды: негізгі жұмыстарға оралмас бұрын оны орындаңыз, себебі олардың BE-сіне шек қалдыратын 2.4 GiB-тан көбірек қажет.

Жұмыс тек `dw_course_l1_streaming.ext_cdc_orders` кестесін қайта жасайды және үш модельденген тапсырысты енгізбес бұрын курстың өз MySQL `course_cdc.orders` кестесін босатады; бұлар WWI тапсырыстары емес, өз сомалары бар. Ол **схемасы бекітілген бір кестені** синхрондайды: бүкіл дерекқорды да, схема өзгерістерін автоматты түрде де емес. Тек курстың демонстрациялық тіркелгілерін қолданыңыз. Қызметтері — `doris-warehousing-streaming` Compose жобасы: жұмыстың соңында тоқтату тәсілі, курстың README файлындағы [Тазалау](../../README.md#clean-up) бөлімінде жою тәсілі берілген.

Жұмысты орындай алмасаңыз да оқыңыз: әр қадамның **Күтілетін нәтижесі** орындалу кезінде не шығатынын көрсетеді.


In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.ui import show_response, show_records
from dw_course.streaming import (
    prepare_streaming, mysql, flink_api, submit_sql, wait_checkpoint, stop_with_savepoint,
    wait_rows, check_flink_job, cleanup_cdc_jobs,
)
prepare_environment(start=True, streaming=True, streaming_profile="cdc")
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("Isolated lab database:", lab.database)

prepare_streaming("cdc", start=True)

<a id="unit-u.6c7fc273806b41d7a195034f75843b7d"></a>

**Күтілетін нәтиже**

`Docker capacity: … GiB RAM, … CPUs`, кемінде 10 GiB және 4 CPU, содан кейін алты жасыл қадамы бар іске қосу панелі (алғаш орындағанда Compose Doris контейнерін 6 GiB шегімен қайта жасайды) және `Isolated lab database: dw_course_l1_streaming`. Compose MySQL, JobManager және TaskManager іске қосады; алғаш орындағанда алдымен Flink бейнесін жинайды. `Flink Web UI: <port>` — Docker 127.0.0.1 үшін таңдаған бос порт: жұмыс кезінде тапсырманы бақылау үшін `http://127.0.0.1:<port>` ашыңыз. MySQL жауап беріп, Flink-те бос тапсырма слоты болғанда күту карточкасы жасыл болады.


<a id="unit-u.61f3ae2d0cae4dadad53f72e026ddfc1"></a>

<a id="1-reset-the-source-and-the-target"></a>

## 1. Дереккөз бен мақсатты кестені қалпына келтіру

Баптау алдыңғы орындау қалдырған `course_mysql_orders` атаулы ескі тапсырмалардан бас тартады; басқа орындалып жатқан Flink тапсырмасына тимейді. Содан кейін MySQL ішіндегі `course_cdc.orders` кестесін босатып, үш тапсырыс енгізеді және Doris ішіндегі `ext_cdc_orders` кестесін Merge-on-Write Unique Key кестесі ретінде қайта жасайды, сондықтан әр тапсырыс — жаңартулар мен жоюлар өзгерте алатын бір жол.

MySQL әр өзгерісті **binlog** ішіне жазады; курстың MySQL жүйесі толық жолдарды (ROW / FULL) жеті күн сақтайды. CDC пайдаланушысы кесте мен binlog-ты оқи алады (`REPLICATION SLAVE`, `REPLICATION CLIENT`). `SHOW MASTER STATUS` MySQL жазған binlog файлы мен орнын көрсетеді: бұл оқырман жеткен орын емес, журналдың соңы.


In [ ]:
cleanup_cdc_jobs()
active = [j for j in flink_api("/jobs/overview")["jobs"] if j["state"] not in {"FINISHED","CANCELED","FAILED"}]
expect(active, [], title="No Flink job of this Lab is running")
mysql("""DELETE FROM course_cdc.orders;
INSERT INTO course_cdc.orders VALUES
(920001,1,'CREATED',100.00),(920002,2,'CREATED',200.00),(920003,3,'CREATED',50.00);
SHOW MASTER STATUS;
SELECT * FROM course_cdc.orders ORDER BY order_id;""")
lab.execute("DROP TABLE IF EXISTS ext_cdc_orders")
lab.execute("""CREATE TABLE ext_cdc_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""");

<a id="unit-u.18f894fb2d0948fd95c9b627b3634bb8"></a>

**Күтілетін нәтиже**

Осы жұмыстың ешбір Flink тапсырмасы орындалмайтынын көрсететін жасыл белгі, содан кейін MySQL-ден екі нәтиже кестесі: `mysql-bin.000003` сияқты `File` және `Position` бар `SHOW MASTER STATUS`, әрі жалпы сомасы 350.00 болатын, бәрі CREATED күйіндегі үш бастапқы тапсырыс 920001, 920002, 920003. Doris қатесі жоқ.


<a id="unit-u.ae0d3a97e8c44819a31c4264081deb63"></a>

<a id="2-submit-the-synchronization-job"></a>

## 2. Синхрондау тапсырмасын жіберу

Flink SQL MySQL үстінде `'scan.startup.mode'='initial'` орнатылған, алдымен snapshot, содан кейін binlog оқитын бастапқы кестені және Doris кестесіне бағытталған қабылдағыш кестені анықтап, `INSERT INTO doris_orders SELECT … FROM mysql_orders` операторын үздіксіз тапсырма ретінде орындайды:

| Параметр | Мағынасы |
| --- | --- |
| `'execution.checkpointing.interval' = '5s'` | Әр 5 секунд сайын тапсырма **checkpoint** сақтайды: binlog орнын және операторларының күйін |
| `'sink.enable-2pc'='true'`, `'sink.label-prefix'` | Қабылдағыш әр checkpoint жолдарын бір Doris транзакциясы ретінде жазып, checkpoint аяқталғанда бекітеді; label-дер әр тәжірибеде жаңа префикстен басталады |
| `'sink.enable-delete'='true'` | MySQL ішіндегі DELETE Doris жолын жояды |
| `'fenodes'='doris:8030'`, `'benodes'='doris:8040'`, `'auto-redirect'='false'` | Flink Doris-ке курс желісіндегі контейнер атауымен қатынап, BE-ге тікелей жүктейді: сынақ ортасының BE-сі 127.0.0.1 ретінде тіркелген, ал Flink контейнерінің ішінде бұл Flink-тің өзі болар еді |

RUNNING күйіндегі тапсырма тек басталғанын білдіреді: келесі қадамдар Doris жолдарын тексереді.


In [ ]:
label = "course_cdc_" + uuid4().hex[:12]
sql = f"""SET 'execution.checkpointing.interval' = '5s';
SET 'parallelism.default' = '1';
SET 'pipeline.name' = 'course_mysql_orders';
CREATE TABLE mysql_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='mysql-cdc','hostname'='mysql','port'='3306',
 'username'='course_cdc','password'='course_cdc_local_only',
 'database-name'='course_cdc','table-name'='orders','server-id'='55101-55102',
 'server-time-zone'='Asia/Shanghai','scan.startup.mode'='initial'
);
CREATE TABLE doris_orders (
 order_id BIGINT, customer_id BIGINT, order_status STRING, order_amount DECIMAL(18,2),
 PRIMARY KEY(order_id) NOT ENFORCED
) WITH (
 'connector'='doris','fenodes'='doris:8030','benodes'='doris:8040','auto-redirect'='false','table.identifier'='dw_course_l1_streaming.ext_cdc_orders',
 'username'='root','password'='', 'sink.label-prefix'='{label}',
 'sink.enable-delete'='true','sink.enable-2pc'='true',
 'sink.properties.format'='json','sink.properties.read_json_by_line'='true'
);
INSERT INTO doris_orders SELECT order_id,customer_id,order_status,order_amount FROM mysql_orders;
"""
print(sql)
job_id = submit_sql(sql)
print("Job ID:", job_id)

<a id="unit-u.c5464bfe31b640ec8ff3261bdff14f80"></a>

**Күтілетін нәтиже**

Шығарылған Flink SQL, содан кейін SQL клиенті журналы: үш `SET` және екі `CREATE TABLE` сәтті болады, INSERT жіберіледі, журнал `Job ID: <32 hexadecimal digits>` жолымен аяқталады. Тапсырма RUNNING болғанда күту карточкасы жасыл болады, ұяшық тапсырма ID-ін қайта шығарады. `jdk.compiler` және қарапайым терминал туралы ескертулер SQL клиентінен келеді, мәні жоқ.


<a id="unit-u.5e3995cf5b344625919451dce23ab61e"></a>

<a id="wait-for-the-snapshot"></a>

### Snapshot-ты күту

Тапсырма алдымен MySQL-де бар жолдарды оқиды. Doris-те сол үш жол болғанша, содан кейін алғашқы аяқталған checkpoint-ке дейін күтіңіз.


In [ ]:
baseline = [[920001,1,"CREATED","100.00"],[920002,2,"CREATED","200.00"],[920003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_cdc_orders",baseline, check_health=lambda: check_flink_job(job_id)),baseline,
       title="The snapshot arrived")
show_records("Completed checkpoint", [(wait_checkpoint(job_id)["latest"]["completed"],)], columns=["Completed checkpoints"])
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="Snapshot results");

<a id="unit-u.bdb4fc453d524d72a2e21e377c2c9895"></a>

**Күтілетін нәтиже**

Күту шамамен 10 секунд ішінде жасыл болады, дереккөздегідей жалпы сомасы 350.00 болатын үш CREATED тапсырысы бар. `Completed checkpoints` кемінде 1: checkpoint әр 5 секунд сайын аяқталады, тек содан кейін қабылдағыш Doris транзакциясын бекітеді.


<a id="unit-u.aea16abdef4b4df6a25c7e1763c97d6d"></a>

<a id="3-change-the-source-and-watch-the-changes-arrive"></a>

## 3. Дереккөзді өзгерту және өзгерістердің жетуін бақылау

MySQL ішінде 920001 күйін PAID етіңіз, 920002-ні жойыңыз және 920004-ті енгізіңіз. Doris ішіндегі жою binlog оқиғасынан келеді: ноутбук Doris-ке DELETE орындамайды. Жалпы сомасы 225.00 болатын үш жол күтіледі.


In [ ]:
mysql("""UPDATE course_cdc.orders SET order_status='PAID' WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920002;
INSERT INTO course_cdc.orders VALUES (920004,4,'CREATED',75.00);
SELECT * FROM course_cdc.orders ORDER BY order_id;
SHOW MASTER STATUS;""")
changed = [[920001,1,"PAID","100.00"],[920003,3,"CREATED","50.00"],[920004,4,"CREATED","75.00"]]
expect(wait_rows(lab,"ext_cdc_orders",changed, check_health=lambda: check_flink_job(job_id)),changed,
       title="The update, the delete, and the insert arrived")
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"225.00"]],
       title="Three orders, 225.00")

<a id="unit-u.e2d56155a7d84d28af45d2b058e0bc86"></a>

**Күтілетін нәтиже**

MySQL 920001 PAID, 920003 және 920004 көрсетеді, `SHOW MASTER STATUS` жоғарырақ `Position` көрсетеді. Шамамен 10 секунд ішінде Doris-те сол үш жол, жалпы 225.00 болады: жаңарту 920001 жолын ауыстырды, ал 920002 жойылды.


<a id="unit-u.4c3189b6fd6643478e293f5fb32bc4c2"></a>

<a id="4-stop-the-job-with-a-savepoint-while-the-source-keeps-changing"></a>

## 4. Дереккөз өзгеріп жатқанда тапсырманы savepoint-пен тоқтату

**Savepoint** — өзіңіз сұрайтын checkpoint: `flink stop` оны жазып, тапсырманы тоқтатады. Ол тапсырманың binlog орнын және операторларының күйін сақтайды. Бұл — апат не автоматты ауыстырып қосу емес, басқарылатын тоқтату мен қалпына келтіру.

Тоқтағаннан кейін дереккөзді тағы үш рет өзгертіңіз. `SHOW MASTER STATUS` алға жылжиды; savepoint өзгермейді, Doris 3-қадамдағы жолдарын сақтайды. Binlog файлдарын, MySQL томын және Flink күй томын жоймаңыз: қалпына келтіруге үшеуі де қажет.


In [ ]:
savepoint = stop_with_savepoint(job_id)
print("Savepoint path:", savepoint)
mysql("""UPDATE course_cdc.orders SET order_amount=125.00 WHERE order_id=920001;
DELETE FROM course_cdc.orders WHERE order_id=920003;
INSERT INTO course_cdc.orders VALUES (920005,5,'CREATED',40.00);
SHOW MASTER STATUS;""")
expect(lab.query("SELECT * FROM ext_cdc_orders ORDER BY order_id"), changed,
       title="Doris keeps the rows of step 3 while the job is stopped")

<a id="unit-u.691a35aaf1034c50a77102e3c79786a1"></a>

**Күтілетін нәтиже**

`Savepoint completed. Path: file:/opt/flink/state/savepoints/savepoint-<6 hex>-<12 hex>` және сол жолы бар `Savepoint path: …`; тапсырма FINISHED болғанда күту жасыл болады. Үш өзгерістен кейін MySQL `SHOW MASTER STATUS` жоғарырақ `Position` көрсетеді, ал Doris-те 3-қадамның үш жолы әлі бар.


<a id="unit-u.bc1f599ac50a4e2fa0bef843cf6edfa1"></a>

<a id="5-restore-from-the-savepoint-and-catch-up"></a>

## 5. Savepoint-тен қалпына келтіру және қуып жету

Тек `'execution.savepoint.path'` қосып, сол SQL-ді жіберіңіз: кестелер, label префиксі және параллельдік бірдей. Дереккөзді де, мақсатты кестені де бастапқы күйге түсірмеңіз. Жаңа тапсырма savepoint binlog орнынан жалғасып, тоқтаған кезде жасалған үш өзгерісті қолданады. Жалпы сомасы 240.00 болатын үш жол күтіңіз және Flink қалпына келтіру жазбасынан savepoint-тен басталғанын тексеріңіз.


In [ ]:
restored_job = submit_sql(f"SET 'execution.savepoint.path' = '{savepoint}';\n" + sql)
final = [[920001,1,"PAID","125.00"],[920004,4,"CREATED","75.00"],[920005,5,"CREATED","40.00"]]
expect(wait_rows(lab,"ext_cdc_orders",final, check_health=lambda: check_flink_job(restored_job)),final,
       title="The job caught up after the restore")
checkpoints = wait_checkpoint(restored_job)
restored = checkpoints["latest"]["restored"]
expect(restored["external_path"],savepoint, title="The job restored from the savepoint")
show_response(restored, title="Restoration record")
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_cdc_orders"),[[3,"240.00"]],
       title="Three orders, 240.00")
mysql("SELECT * FROM course_cdc.orders ORDER BY order_id")
lab.sql("SELECT * FROM ext_cdc_orders ORDER BY order_id", title="Results after restoration");

<a id="unit-u.a97f70bf6e02453c8da3e01208808899"></a>

**Күтілетін нәтиже**

Жаңа тапсырма ID-і, содан кейін үш жол: 920001 PAID 125.00, 920004 CREATED 75.00 және 920005 CREATED 40.00; барлығы 240.00, 920003 жойылған. Қалпына келтіру жазбасында `is_savepoint` True және savepoint жолына тең `external_path` бар. MySQL-дің екі нәтиже жиыны Doris-пен сәйкес келеді.


<a id="unit-u.0312655cb2fc48829310eb17bcdb7774"></a>

<a id="6-practice-restore-from-a-newer-savepoint"></a>

## 6. Жаттығу: жаңарақ savepoint-тен қалпына келтіру

Қалпына келтірілген тапсырманы жаңа savepoint-пен тоқтатыңыз, ол тоқтап тұрғанда MySQL ішіндегі 920004 тапсырысын PAID етіңіз, жаңа savepoint-тен қалпына келтіріңіз және Doris қуып жеткенін, Flink жаңа жолдан қалпына келгенін тексеріңіз. Жаңа тапсырма ID-ін `restored_job` ішінде сақтаңыз: соңғы ұяшық сол айнымалы атаған тапсырманы тоқтатады.

Содан кейін MySQL ішіндегі соңғы өзгеріс уақыты неге savepoint орнына жүрмейтінін түсіндіріңіз.


In [ ]:
# Your code here

<a id="unit-u.634a1afa8884473aa03476c3f22d2761"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
practice_savepoint = stop_with_savepoint(restored_job)
mysql("UPDATE course_cdc.orders SET order_status='PAID' WHERE order_id=920004;")
practice_job = submit_sql(f"SET 'execution.savepoint.path' = '{practice_savepoint}';\n" + sql)
practice_rows = [[920001,1,"PAID","125.00"],[920004,4,"PAID","75.00"],[920005,5,"CREATED","40.00"]]
expect(wait_rows(lab, "ext_cdc_orders", practice_rows, check_health=lambda: check_flink_job(practice_job)),
       practice_rows, title="Doris caught up with the change made while the job was stopped")
expect(wait_checkpoint(practice_job)["latest"]["restored"]["external_path"], practice_savepoint,
       title="The job restored from the new savepoint")
restored_job = practice_job  # the last cell stops this job

<a id="unit-u.e91f19b908f44a9782ac3749b716d044"></a>

**Күтілетін нәтиже**

Savepoint жолы, жаңа тапсырма ID-і және Doris ішінде 920001 PAID 125.00, 920004 PAID 75.00, 920005 CREATED 40.00; қалпына келтірілген checkpoint — жаңа savepoint.

Тапсырыс соңғы рет өзгерген уақыт сияқты бизнес уақыты savepoint-ті алмастыра алмайды. Savepoint тапсырма оқыған binlog файлы мен орнын және операторларының күйін, оның ішінде қабылдағыштың күтіп тұрған Doris транзакцияларын тіркейді. Қолданба жазған уақыт орынды анықтамайды: бірнеше өзгеріс бір секундта болуы мүмкін, жойылған жолда оқитын уақыт қалмайды, кеш бекітілген транзакцияда ертерек уақыт болуы мүмкін. Мұндай уақыттан қайта бастау өзгерістерді өткізіп жібереді не екі рет қолданады.


<a id="unit-u.93055a867572454e8b81caec090deb25"></a>

<a id="7-stop-the-job-and-keep-the-results"></a>

## 7. Тапсырманы тоқтату және нәтижелерді сақтау

Орындалып жатқан тапсырманы соңғы savepoint-пен тоқтатып, қосылымды жабыңыз. Doris жолдарын сақтайды, savepoint-тар Flink күй томында қалады, сондықтан шығарылған жолмен бұл тапсырманы кейін қалпына келтіре аласыз. Бұл жұмыс қалыпты жұмысты және басқарылатын қалпына келтіруді тексереді; әр сәтсіздіктегі exactly-once мінез-құлқын, ретсіз келетін оқиғаларды, бірнеше кесте арасындағы көрінуді не өндірістік жүктемені көрсетпейді.


In [ ]:
final_savepoint = stop_with_savepoint(restored_job)
print("Final savepoint path:",final_savepoint)
lab.close()

<a id="unit-u.292020899238453b8f6a32535f63a853"></a>

**Күтілетін нәтиже**

`Final savepoint path: file:/opt/flink/state/savepoints/savepoint-…`. Енді Flink тапсырмасы орындалмайды; MySQL мен Flink сіз тоқтатқанша жұмыс істейді.


<a id="unit-u.cceadf0587fa48e38e32a3a8aa3c3a6c"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- MySQL snapshot-ының, жаңартудың, жоюдың және енгізудің Doris-ке жеткенін көрдіңіз;
- checkpoint пен savepoint нені сақтайтынын және `SHOW MASTER STATUS` неге тапсырманың орны емес екенін түсіндіре аласыз;
- тапсырманы savepoint-тен қалпына келтіріп, тоқтаған кезде жасалған өзгерістерді қуып жеткенін тексердіңіз.

MySQL мен Flink-ті тоқтатып, деректері мен savepoint-тарын сақтау үшін курс каталогында мынаны орындаңыз:

```bash
docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml --profile cdc stop
```

Оларды қайта іске қосу тоқтатылған Flink тапсырмасын жалғастырмайды: жұмыс шығарған savepoint жолымен SQL-ін қайта жіберіңіз.

Негізгі жұмыстарға оралмас бұрын 1-зертханалық жұмыстың **Іске қосу және қосылу** ұяшығын орындаңыз: ол Doris контейнерін 6 GiB шектеуінсіз қайта жасап, деректерін сақтайды.

**Бірдеңе дұрыс болмаса:**

- Бірінші ұяшық іске қосудан бас тартты: Docker-ге көрсетілген жады мен CPU санын беріңіз.
- `MEM_LIMIT_EXCEEDED` не `MEM_ALLOC_FAILED`: 1-зертханалық жұмыстың іске қосу ұяшығы Doris-тің 6 GiB шегін алып тастаған болуы мүмкін. Басқа ноутбуктерді аяқтап, осы жұмыстың бірінші ұяшығын қайта орындаңыз, ол шекті қайта орнатады; сонда `docker inspect doris-warehousing-course-doris-1 --format '{{.HostConfig.Memory}}'` 6442450944 көрсетеді.
- Flink тапсырмасы сәтсіз болғандықтан күту тоқтады: Flink web UI ішіндегі **Exceptions** қойындысын ашып, журналдарды `docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml logs --tail 100 jobmanager taskmanager mysql` арқылы оқыңыз.
- Жұмыс үзілді: бірінші ұяшықтан қайта орындаңыз. Ол осы жұмыстың бұрынғы орындауы қалдырған `course_mysql_orders` тапсырмасынан бас тартады және басқа Flink тапсырмасынан бас тартпайды. Оның орнына savepoint сақтағыңыз келсе, 4-қадамдағыдай `flink stop` арқылы тоқтатыңыз.
- Қалпына келтіру қуып жетпеді: web UI ішіндегі тапсырма қателері мен checkpoint-тарын тексеріңіз. Flink күйі мен MySQL томдарын сақтаңыз: қалпына келтіруге savepoint және ол сілтейтін binlog файлдары қажет.
Іске қосу және қосылу сынақ ортасының бұрынғы деректер көлі, бақылау тақтасы және сақтық көшірме желілік қосылымдарын да қалпына келтіреді. Ол қызметтерді бұрын жасасаңыз, көл сұрауы мен сақталған Superset диаграммалары әлі жүктелетінін тексеріңіз; деректері мен рұқсаттары сақталады.


<a id="unit-u.a5a447208e664af69ba7d9efa4ac134e"></a>

<a id="next"></a>

## Келесі қадам

Әлі тапсырмасаңыз, [үздіксіз жүктеу тестін](quiz05_continuous_ingestion.ipynb) тапсырыңыз немесе [5-модуль сабағына](course.md) оралыңыз. 6-модуль [6-модуль сабағынан](../module06-data-quality/course.md) басталады; 7-модуль үзілістен кейін тапсырыс оқиғаларын қайта ойнатады.
